# This Section Concerns all mentioned Datasets with Boost

# Imports

In [ ]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse

# ML utilities
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve
)

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier


def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result

ModuleNotFoundError: No module named 'xgboost'

In [2]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


## XGBoost Pipeline

In [3]:
boost_powers = [2**i for i in range(16)]  # 1 ... 32768

BST_DT_ITERS   = [t for t in boost_powers if t <= 1024]   # trees stop at 2^10
BST_STMP_ITERS = boost_powers                             # stumps allow full 2^15

def train_xgb_boost(
    X_train, y_train, X_val, y_val,
    n_estimators,
    max_depth,
    min_child_weight=50,
    learning_rate=0.1,
):
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,   # ID3-style stopping rule
        subsample=1.0,
        colsample_bytree=1.0,
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        objective="binary:logistic",
        eval_metric="logloss",
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    return acc, model

from sklearn.model_selection import train_test_split

def select_best_boosting_iterations(
    X_train, y_train,
    iteration_list,
    family_name,
    max_depth,
):
    print(f"\n### Phase 1 — Selecting best boosting rounds for {family_name}")

    # Step 1: Split training into train-subset and validation-subset
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    results = []
    best_acc = -1
    best_iters = None

    for iters in iteration_list:
        acc, _ = train_xgb_boost(
            X_tr, y_tr, X_val, y_val,
            n_estimators=iters,
            max_depth=max_depth,
        )
        results.append((iters, acc))
        print(f"  Boosting rounds {iters:5d} → val acc = {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_iters = iters

    print(f"\nSelected boosting rounds for {family_name}: {best_iters}")
    return best_iters, pd.DataFrame(results, columns=["boost_rounds", "val_acc"])



def train_final_boosted_model(
    X_train, y_train,
    best_iters,
    max_depth,
    family_name,
):
    print(f"\n### Phase 2 — Training final {family_name} model on full training set")

    final_model = XGBClassifier(
        n_estimators=best_iters,
        max_depth=max_depth,
        learning_rate=0.1,
        min_child_weight=50,
        subsample=1.0,
        colsample_bytree=1.0,
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        objective="binary:logistic",
        eval_metric="logloss",
    )

    final_model.fit(X_train, y_train)
    return final_model


def evaluate_final_model(
    model, X_test, y_test,
    family_name,
    best_iters
):
    print(f"\n### Phase 3 — Final Evaluation on Test Set ({family_name})")
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    cm  = confusion_matrix(y_test, preds)

    print(f"Boosting rounds used: {best_iters}")
    print(f"Test Accuracy: {acc:.4f}")
    print("Confusion Matrix:\n", cm)

    return {
        "family": family_name,
        "best_iters": best_iters,
        "test_acc": acc,
        "test_cm": cm
    }

def full_boosting_family_pipeline(
    X_train, y_train, X_test, y_test,
    iteration_list,
    family_name,
    max_depth
):
    # Phase 1 — select best iteration count
    best_iters, selection_table = select_best_boosting_iterations(
        X_train, y_train,
        iteration_list,
        family_name=family_name,
        max_depth=max_depth,
    )

    # Phase 2 — retrain final boosted model
    final_model = train_final_boosted_model(
        X_train, y_train,
        best_iters=best_iters,
        max_depth=max_depth,
        family_name=family_name,
    )

    # Phase 3 — evaluate on test set
    results = evaluate_final_model(
        final_model, X_test, y_test,
        family_name=family_name,
        best_iters=best_iters
    )

    results["selection_table"] = selection_table
    results["model"] = final_model
    return results


# 30 70 Split

In [ ]:
wine_3070 = all_splits["3070"]["wine"]


X_train = wine_3070["X_train"].to_numpy().astype("float32")
X_test  = wine_3070["X_test"].to_numpy().astype("float32")
y_train = wine_3070["y_train"].to_numpy().astype("int32")
y_test  = wine_3070["y_test"].to_numpy().astype("int32")

BST_DT_ITERS = [2**i for i in range(11)]  # 1..1024

bst_dt = full_boosting_family_pipeline(
    X_train, y_train, X_test, y_test,
    iteration_list=BST_DT_ITERS,
    family_name="BST-DT",
    max_depth=20
)

BST_STMP_ITERS = [2**i for i in range(16)]  # 1..32768

bst_stmp = full_boosting_family_pipeline(
    X_train, y_train, X_test, y_test,
    iteration_list=BST_STMP_ITERS,
    family_name="BST-STMP",
    max_depth=1
)




### Phase 1 — Selecting best boosting rounds for BST-DT


NameError: name 'XGBClassifier' is not defined

In [ ]:
cup98_3070    = all_splits["3070"]["cup98"]
customer_3070 = all_splits["3070"]["customer"]

# 70 30 Split

In [ ]:
wine_7030     = all_splits["7030"]["wine"]
cup98_7030    = all_splits["7030"]["cup98"]
customer_7030 = all_splits["7030"]["customer"]


# 50 50 Split

In [ ]:
wine_5050     = all_splits["5050"]["wine"]
cup98_5050    = all_splits["5050"]["cup98"]
customer_5050 = all_splits["5050"]["customer"]
